In [11]:
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
import os

# Take environment variables from .env
load_dotenv(override=True)

credential = DefaultAzureCredential()

bearer_token_provider = get_bearer_token_provider(
    credential, "https://management.azure.com/.default"
)

In [12]:
FOUNDRY_ENDPOINT = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
MODEL_DEPLOYMENT = os.environ.get("FOUNDRY_MODEL_DEPLOYMENT_NAME")

In [ ]:
import requests

# Safe diagnostic: list agents without following redirects so we can inspect proxy/VPN behavior.
token = credential.get_token("https://ai.azure.com/.default").token
agents_url = f"{FOUNDRY_ENDPOINT.rstrip('/')}/agents?api-version=v1"

response = requests.get(
    agents_url,
    headers={
        "Authorization": f"Bearer {token}",
        "Accept": "application/json",
    },
    allow_redirects=False,
    timeout=30,
)

print(f"Request URL: {agents_url}")
print(f"Status: {response.status_code} {response.reason}")
print(f"Location: {response.headers.get('Location')}")
print(f"Content-Type: {response.headers.get('Content-Type')}")
print("Body preview:")
print(response.text[:1000])

In [13]:
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, MCPTool

project_client = AIProjectClient(
    endpoint=FOUNDRY_ENDPOINT, credential=credential
)

# Agent instructions optimized for knowledge retrieval with citations
instructions = """
 You are an assistant. 
"""


agent = project_client.agents.create_version(
    agent_name="foundry-test-agent",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT,
        instructions=instructions
    ),
)

print(f"✅ Agent '{agent.name}' created (version={agent.version})")

Ran into a deserialization error. Ignoring since this is failsafe deserialization
Traceback (most recent call last):
  File "c:\develop\pythonenv\venv_py312\Lib\site-packages\azure\ai\projects\_utils\model_base.py", line 1065, in _failsafe_deserialize
    return _deserialize(deserializer, response.json(), module, rf, format)
                                      ^^^^^^^^^^^^^^^
  File "c:\develop\pythonenv\venv_py312\Lib\site-packages\azure\core\rest\_http_response_impl.py", line 331, in json
    self._json = loads(self.text())
                 ^^^^^^^^^^^^^^^^^^
  File "C:\Users\jbao6\AppData\Local\Programs\Python\Python312\Lib\json\__init__.py", line 346, in loads
    return _default_decoder.decode(s)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\jbao6\AppData\Local\Programs\Python\Python312\Lib\json\decoder.py", line 337, in decode
    obj, end = self.raw_decode(s, idx=_w(s, 0).end())
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\jbao6\AppData\Local

HttpResponseError: Operation returned an invalid status 'Moved Temporarily'